# 03 · Update, measure, gate, promote

*Part 2, sections 3-4.* The first full pass of the loop, by hand: a person writes v2, both versions
run on the same dataset with the same scorers, and v2 ships only if it measurably helps.

The rules every later level reuses (`selfimprove/gate.py`):

1. **must-pass** scorers are at 100% for the candidate, on both splits
2. the candidate is **not worse** than production on the train split (the CI suite)
3. it **beats production on the holdout** by at least `SI_PROMOTION_MARGIN`
4. it is better somewhere (a no-op never ships), and not bloated

In [1]:
import _nb as nb

experiment_id = nb.setup()

/Users/jon/Documents/Claude/ODSC West 2026/self-improving-agent/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


env file      /Users/jon/Documents/Claude/ODSC West 2026/self-improving-agent/harness/.env
tracking      sqlite:////Users/jon/Documents/Claude/ODSC West 2026/self-improving-agent/harness/.selfimprove/notebooks/mlflow.db   experiment walkthrough
agent         python: examples.support_agent:answer
instructions  prompts:/walkthrough-instructions  (seed /Users/jon/Documents/Claude/ODSC West 2026/self-improving-agent/harness/examples/instructions.md)
judge         anthropic:/claude-opus-5-5  (credentials found)
first pass    typesafe:/jev-latest  (re-judge p(pass) in 0.50-0.80)
improver      analyst   auto-promote False
mode          live


In [2]:
import mlflow
from selfimprove import agents, config, gate, instructions, signals, store

print(open("examples/fixes/v2_manual.md").read())

# Acme Cloud Support Agent

You are the support agent for Acme Cloud. You answer customer questions about
billing, account access, and how to use the product.

## How to answer
- Put the direct answer in your first sentence. Add conditions or steps after it.
- Always call `lookup_policy` before answering, with the topic that matches the
  question: billing, account_access, or product_howto. The policy is the source of
  truth and overrides anything you remember.
- If the policy does not cover the question, say so and offer to escalate to a human
  teammate. Never guess at plan limits, timelines, or features.
- Keep answers under 120 words.



## 1. Register v2 without promoting it

The commit message names the evidence. `production` stays on v1 until v2 has been measured.

In [3]:
v2 = mlflow.genai.register_prompt(
    name=config.settings.prompt_name,
    template=open("examples/fixes/v2_manual.md").read(),
    commit_message="v2: lead with the answer, always look up the policy, never guess "
                   "(evidence: the stale-refund and disclaimer failures in notebook 01)",
    tags={"author": "you", "status": "candidate"},
)
print(instructions.diff("production", v2.version))

--- v1
+++ v2
@@ -4,14 +4,10 @@
 billing, account access, and how to use the product.
 
 ## How to answer
-- Be warm and thorough. Start by reminding the customer that policies can vary by
-  plan and region and that they should always double-check the Terms of Service.
-- Answer quickly from the quick facts below. Looking things up slows you down, so
-  only call `lookup_policy` if you really have to.
-- Keep the customer happy. If you are not sure, give your best guess.
-
-## Quick facts
-- Refunds: customers can get a refund within 30 days of purchase.
-- Password reset links are valid for 24 hours.
-- Data export: Settings → Data → Export.
-- Teammates: Workspace → Members → Invite.
+- Put the direct answer in your first sentence. Add conditions or steps after it.
+- Always call `lookup_policy` before answering, with the topic that matches the
+  question: billing, account_access, or product_howto. The policy is the source of
+  truth and overrides anything you remember.
+- If the 

**Under the hood:** this is what the harness's `instructions.register_candidate()` does.
`instructions.diff()` diffs the two texts in Python.

**MLflow in this step**

* `mlflow.genai.register_prompt()` with an existing name: adds version 2 under the same prompt, with
  its own tags (`author`, `status=candidate`). No alias moves, so `@production` still resolves to v1
  and live traffic is unaffected
* `mlflow.genai.load_prompt()`: loads both versions for the diff

In [4]:
for v in (1, v2.version):
    p = mlflow.genai.load_prompt(f"prompts:/{config.settings.prompt_name}/{v}")
    print(f"v{p.version}  aliases={p.aliases}  tags={p.tags}")

v1  aliases=['production']  tags={}
v2  aliases=[]  tags={'author': 'you', 'status': 'candidate'}


## 2. Measure both on the same data

Each `evaluate()` call is an MLflow evaluation run, so the two sit side by side in the UI with
per-row outcomes and rationales. `predict_fn` pins a version: the agent answers every row fresh.

In [5]:
train, holdout = store.rows("train"), store.rows("holdout")
scorers = signals.full()
print(len(train), "train rows,", len(holdout), "holdout rows;", [s.name for s in scorers])

v1_train = mlflow.genai.evaluate(data=train, predict_fn=agents.predict_fn(1), scorers=scorers)
v2_train = mlflow.genai.evaluate(data=train, predict_fn=agents.predict_fn(v2.version), scorers=scorers)

7 train rows, 8 holdout rows; ['no_stale_policy', 'no_disclaimer_opener', 'looked_up_policy', 'concise', 'grounded_in_policy', 'leads_with_answer', 'no_guessing', 'quality', 'has_key_facts', 'correctness']


2026/10/28 13:01:48 INFO mlflow.models.evaluation.utils.trace: Auto tracing is temporarily enabled during the model evaluation for computing some metrics and debugging. To disable tracing, call `mlflow.autolog(disable=True)`.
2026/10/28 13:01:48 INFO mlflow.genai.utils.data_validation: Testing model prediction with the first sample in the dataset. To disable this check, set the MLFLOW_GENAI_EVAL_SKIP_TRACE_VALIDATION environment variable to True.
Evaluating: 100%|██████████| 7/7 [Elapsed: 00:16, Remaining: 00:00] [predict_fn: 36%, scorers: 64%]
2026/10/28 13:02:08 INFO mlflow.genai.utils.data_validation: Testing model prediction with the first sample in the dataset. To disable this check, set the MLFLOW_GENAI_EVAL_SKIP_TRACE_VALIDATION environment variable to True.



✨ Evaluation completed.

Metrics and evaluation results are logged to the MLflow run:
  Run name: legendary-bear-955
  Run ID: 11ee3ecc255a4a6dae263722c7e72695

To view the detailed evaluation results with sample-wise scores,
open the Traces tab in the Run page in the MLflow UI.



Evaluating: 100%|██████████| 7/7 [Elapsed: 00:13, Remaining: 00:00] [predict_fn: 31%, scorers: 69%]


✨ Evaluation completed.

Metrics and evaluation results are logged to the MLflow run:
  Run name: sneaky-sheep-205
  Run ID: 7d7075f1f0864678b76a38ebdb33ffcc

To view the detailed evaluation results with sample-wise scores,
open the Traces tab in the Run page in the MLflow UI.



In [6]:
import pandas as pd

def pass_rates(result):
    df = result.result_df
    return {s.name: df[f"{s.name}/value"].map(gate.passed).mean() for s in scorers if f"{s.name}/value" in df}

pd.DataFrame({"v1": pass_rates(v1_train), "v2": pass_rates(v2_train)}).style.format("{:.0%}")

,v1,v2
no_stale_policy,57%,100%
no_disclaimer_opener,29%,100%
looked_up_policy,71%,100%
concise,71%,100%
grounded_in_policy,0%,57%
leads_with_answer,14%,100%
no_guessing,86%,86%
quality,0%,86%
has_key_facts,100%,100%
correctness,57%,100%


**Under the hood:** `agents.predict_fn(v)` wraps `agents.run(question, ref=v, source="eval")`, so
every answer is a new trace tagged `si_source=eval` and the pinned `si_prompt_version`.

**MLflow in this step**

* `mlflow.genai.evaluate(data=rows, predict_fn=..., scorers=...)`: calls the agent once per row, logs
  the scorers' verdicts on each new trace, and writes the per-scorer averages as run metrics
* trace metadata `mlflow.sourceRun`: links each of those traces to the evaluation run

In [7]:
evals = mlflow.search_traces(run_id=v2_train.run_id, return_type="list")
print(len(evals), "traces in the v2 run; si_source:", {t.info.tags.get("si_source") for t in evals},
      "| si_prompt_version:", {t.info.tags.get("si_prompt_version") for t in evals})
print("linked to the run:", all(t.info.trace_metadata.get("mlflow.sourceRun") == v2_train.run_id for t in evals))
print({k: round(float(v), 2) for k, v in v2_train.metrics.items()})

7 traces in the v2 run; si_source: {'eval'} | si_prompt_version: {'2'}
linked to the run: True
{'looked_up_policy/mean': 1.0, 'concise/mean': 1.0, 'no_stale_policy/mean': 1.0, 'no_disclaimer_opener/mean': 1.0, 'has_key_facts/mean': 1.0, 'grounded_in_policy/mean': 0.57, 'correctness/mean': 1.0, 'leads_with_answer/mean': 1.0, 'quality/mean': 0.86, 'no_guessing/mean': 1.0}


Three things to check before trusting the numbers: the cases that failed should now pass (the
change targeted them), the golden cases should still pass, and the **raw responses** should
actually be good, not just pass:

In [8]:
for _, row in v2_train.result_df.head(4).iterrows():
    print("Q:", row["request"]["question"])
    print("A:", str(row["response"])[:240], "\n")

Q: How do I export all my data?
A: You can export all your data from **Settings → Data → Export**, choosing CSV or JSON as the format. Large workspaces can take up to 24 hours to process, and you'll get an email when the export is ready. 

Q: I bought Pro three weeks ago and barely used it. Can I get a partial refund?
A: No, I can't offer a partial refund for Pro. Refunds are only available within 14 days of purchase, and prorated refunds aren't issued after that window. At three weeks, your purchase is outside it.

If you want to cancel going forward, you  

Q: How long is a password reset link valid?
A: A password reset link is valid for 15 minutes after it is sent. If it expires, request a new one and use it within 15 minutes. If you still can't log in after resetting, clear your browser cache and try again. 

Q: What happens if my card payment fails?
A: If a card payment fails, Acme Cloud retries it 3 times over 7 days. If the third attempt also fails, your workspace is downgraded 

## 3. The gate

`gate.compare()` does the above for both splits and applies the rules. It is the same function
that gates the analyst's, the optimizer's and a coding agent's candidates later.

In [9]:
decision = gate.compare(v2.version, "production")
print(decision.table())
print()
print(decision.summary())

2026/10/28 13:02:24 INFO mlflow.genai.utils.data_validation: Testing model prediction with the first sample in the dataset. To disable this check, set the MLFLOW_GENAI_EVAL_SKIP_TRACE_VALIDATION environment variable to True.
Evaluating: 100%|██████████| 7/7 [Elapsed: 00:16, Remaining: 00:00] [predict_fn: 34%, scorers: 66%]
2026/10/28 13:02:43 INFO mlflow.genai.utils.data_validation: Testing model prediction with the first sample in the dataset. To disable this check, set the MLFLOW_GENAI_EVAL_SKIP_TRACE_VALIDATION environment variable to True.



✨ Evaluation completed.

Metrics and evaluation results are logged to the MLflow run:
  Run name: train/baseline
  Run ID: f782d1ad5f1f491889741ebf4f1eb271

To view the detailed evaluation results with sample-wise scores,
open the Traces tab in the Run page in the MLflow UI.



Evaluating: 100%|██████████| 7/7 [Elapsed: 00:13, Remaining: 00:00] [predict_fn: 35%, scorers: 65%]
2026/10/28 13:02:59 INFO mlflow.genai.utils.data_validation: Testing model prediction with the first sample in the dataset. To disable this check, set the MLFLOW_GENAI_EVAL_SKIP_TRACE_VALIDATION environment variable to True.



✨ Evaluation completed.

Metrics and evaluation results are logged to the MLflow run:
  Run name: train/candidate
  Run ID: eed50f6524a444f8bd462114a56c635a

To view the detailed evaluation results with sample-wise scores,
open the Traces tab in the Run page in the MLflow UI.



Evaluating: 100%|██████████| 8/8 [Elapsed: 00:16, Remaining: 00:00] [predict_fn: 44%, scorers: 56%]
2026/10/28 13:03:20 INFO mlflow.genai.utils.data_validation: Testing model prediction with the first sample in the dataset. To disable this check, set the MLFLOW_GENAI_EVAL_SKIP_TRACE_VALIDATION environment variable to True.



✨ Evaluation completed.

Metrics and evaluation results are logged to the MLflow run:
  Run name: holdout/baseline
  Run ID: 1ea168436c304bf88e26254d37157c39

To view the detailed evaluation results with sample-wise scores,
open the Traces tab in the Run page in the MLflow UI.



Evaluating: 100%|██████████| 8/8 [Elapsed: 00:14, Remaining: 00:00] [predict_fn: 34%, scorers: 66%]



✨ Evaluation completed.

Metrics and evaluation results are logged to the MLflow run:
  Run name: holdout/candidate
  Run ID: 21b32e2a5b724da5b7b2cd9a4d3d201c

To view the detailed evaluation results with sample-wise scores,
open the Traces tab in the Run page in the MLflow UI.

scorer                         train/baseline    train/candidate   holdout/baseline  holdout/candidate
concise                                   57%               100%                75%               100%
correctness                               43%               100%                88%                88%
grounded_in_policy                         0%                43%                 0%                75%
has_key_facts                            100%               100%               100%                88%
leads_with_answer                         29%               100%                50%               100%
looked_up_policy                          71%               100%               100%               100

**Under the hood:** `gate.compare()` writes one decision record in MLflow: a parent run with four
nested evaluation runs, and a verdict on the candidate prompt version.

**MLflow in this step**

* `mlflow.start_run(run_name="gate: v2 vs v1")` and `mlflow.set_tags()`: the parent run, tagged
  `si.job=gate`, `si.candidate_version`, `si.baseline_version`
* `mlflow.start_run(run_name="<split>/<role>", nested=True)`: four nested runs, `train/baseline`,
  `train/candidate`, `holdout/baseline`, `holdout/candidate`
* `mlflow.genai.evaluate(data=rows, predict_fn=agents.predict_fn(version), scorers=...)`: one per nested
  run, which is then tagged `si.label` and `si.prompt_version`
* `mlflow.log_metric()`: `<split>_<role>_overall` and `prompt_growth` on the parent
* `mlflow.set_tags()`: `si.gate_passed` and `si.gate_reasons` on the parent
* `mlflow.log_text()`: the artifacts `gate_table.txt` and `candidate.diff`
* `mlflow.genai.set_prompt_version_tag()`: `gate=passed|failed` and `gate_run=<run id>` on the
  candidate prompt version

In [10]:
gate_run = mlflow.get_run(decision.run_id)
print(gate_run.info.run_name, {k: v for k, v in gate_run.data.tags.items() if k.startswith("si.")})
print({k: round(v, 2) for k, v in gate_run.data.metrics.items()})
for r in mlflow.search_runs(filter_string=f"tags.mlflow.parentRunId = '{decision.run_id}'", output_format="list"):
    print(f"  nested {r.info.run_name:<18} si.label={r.data.tags.get('si.label')}  si.prompt_version={r.data.tags.get('si.prompt_version')}")
print("artifacts:", [f.path for f in mlflow.MlflowClient().list_artifacts(decision.run_id)])
cand = mlflow.genai.load_prompt(f"prompts:/{config.settings.prompt_name}/{decision.candidate}")
print("prompt version tags:", {k: v for k, v in cand.tags.items() if k.startswith("gate")})

gate: v2 vs v1 {'si.job': 'gate', 'si.candidate_version': '2', 'si.baseline_version': '1', 'si.gate_passed': 'True', 'si.gate_reasons': ''}
{'train_baseline_overall': 0.44, 'train_candidate_overall': 0.93, 'holdout_baseline_overall': 0.6, 'holdout_candidate_overall': 0.85, 'prompt_growth': 0.86}
  nested holdout/candidate  si.label=holdout/candidate  si.prompt_version=2
  nested holdout/baseline   si.label=holdout/baseline  si.prompt_version=1
  nested train/candidate    si.label=train/candidate  si.prompt_version=2
  nested train/baseline     si.label=train/baseline  si.prompt_version=1
artifacts: ['candidate.diff', 'gate_table.txt']
prompt version tags: {'gate': 'passed', 'gate_run': 'a878dffe70cd44789ac3662e16e09b27'}


## 4. The dataset as a regression suite

The same rows run as a pytest suite, one MLflow run per session, and CI runs it on every pull request
(`.github/workflows/selfimprove-regression.yml`).

**MLflow in this step**

* `@mlflow.test`: marks a test for MLflow's pytest plugin, which manages its runs and traces
* `-p mlflow.pytest.plugin`: turns the plugin on; it opens one MLflow run per session
* `mlflow.genai.evaluate(data=[row], predict_fn=..., scorers=...)`: what each test runs on its dataset row

The test, one per dataset row:

In [11]:
print(open("tests/regression/test_regressions.py").read().split('"""', 2)[2])



from __future__ import annotations

import os

import mlflow
import pytest

if os.getenv("SI_REGRESSION") != "1":
    pytest.skip("set SI_REGRESSION=1 to run the regression suite (calls the agent and judges)",
                allow_module_level=True)

from selfimprove import agents, gate, signals, store, traces  # noqa: E402

traces.setup()
REF = os.getenv("SI_PROMPT_REF", "production")
ROWS = store.records()
SCORERS = signals.full()


def _id(r: dict) -> str:
    q = store.question_of(r)[:40]
    return f"{r['tags'].get('split', 'train')}:{r['tags'].get('theme') or r['tags'].get('source', '')}:{q}"


@mlflow.test
@pytest.mark.parametrize("row", ROWS, ids=[_id(r) for r in ROWS])
def test_dataset_row(row):
    result = mlflow.genai.evaluate(data=[{"inputs": row["inputs"], "expectations": row["expectations"]}],
                                   predict_fn=agents.predict_fn(REF), scorers=SCORERS)
    failures = gate.scorer_failures(result, SCORERS)
    assert not failures, "\n".join(fa

```bash
SI_REGRESSION=1 pytest -p mlflow.pytest.plugin tests/regression -n 4
```

## 5. Promote: a pointer move

The agent picks up v2 on its next request, with no code change. The version it replaced becomes
`previous`, so rollback is one alias move too.

In [12]:
if decision.passed:
    replaced = instructions.promote(v2.version, reason=f"gate run {decision.run_id}", by="you")
    print(f"production: v{replaced} -> v{instructions.load('production').version}")
else:
    print("not promoting:", decision.reasons)

production: v1 -> v2


**Under the hood:** `instructions.promote()` moves two aliases and records why on both versions. The
agent resolves `@production` on its next call, which is why no code changes. `instructions.rollback()`
is the reverse alias move.

**MLflow in this step**

* `mlflow.genai.set_prompt_alias()`: first `previous` to the old version, then `production` to the new
  one
* `mlflow.genai.set_prompt_version_tag()`: the new version gets `status=production`, `promoted_by`,
  `promoted_because`, `promoted_at` and `replaced_version`; the old one gets `status=retired`

In [13]:
for v in (1, v2.version):
    p = mlflow.genai.load_prompt(f"prompts:/{config.settings.prompt_name}/{v}")
    print(f"v{p.version}  aliases={p.aliases}  " + str({k: p.tags.get(k) for k in ("status", "promoted_by", "replaced_version")}))

v1  aliases=['previous']  {'status': 'retired', 'promoted_by': None, 'replaced_version': None}
v2  aliases=['production']  {'status': 'production', 'promoted_by': 'you', 'replaced_version': '1'}


In [14]:
print(agents.run("How long can I wait to request a refund?", source="production"))

You can request a refund within **14 days of purchase**. This applies to all plans (Monthly, Annual, Pro, and Enterprise). Refunds aren't prorated after the 14-day window.

To request one, go to Settings → Billing → Request refund.


One pass produced two prompt versions, a dataset, two evaluation runs that show the before and
after, and a regression suite. Every step was manual.

**Next:** [04 · Triage and approve](04_triage_and_approve.ipynb), for when there is too much signal
to read by hand.